# LeetCode #29 — Divide Two Integers: chia bằng phép trừ và nhân đôi (bit shift)

**Mức độ:** Medium · **Chủ đề:** Math, Bit Manipulation

Skeleton trên LeetCode:

```python
class Solution:
    def divide(self, dividend: int, divisor: int) -> int:
```

Đề cấm dùng phép nhân, chia và lấy dư. Ta phải **tự xây phép chia** từ phép trừ, phép cộng và phép dịch bit.

Mục tiêu của notebook:

1. Hiểu chia "làm tròn về 0" (truncate) khác `//` của Python thế nào.
2. Chia bằng **trừ lặp** — đúng nhưng quá chậm.
3. Tăng tốc bằng **nhân đôi số chia** (`<< 1`) — ý tưởng giống "chia dài" (long division) trong hệ nhị phân.
4. Xử lý **dấu** và **trường hợp tràn duy nhất**: `-2^31 / -1`.

# 1. Bài toán là gì?

Cho hai số nguyên `dividend` (số bị chia) và `divisor` (số chia). Chia hai số **không dùng** `*`, `/`, `%`.

Phép chia **làm tròn về 0**: bỏ phần thập phân. Ví dụ `8.345 → 8`, `-2.7335 → -2`.

Môi trường chỉ chứa số nguyên 32-bit có dấu `[-2^31, 2^31 - 1]`. Nếu thương **lớn hơn** `2^31 - 1` thì trả `2^31 - 1`; nếu **nhỏ hơn** `-2^31` thì trả `-2^31`.

**Constraints (theo đề):**

- `-2^31 <= dividend, divisor <= 2^31 - 1`
- `divisor != 0`

---

## Ví dụ

```text
dividend = 10, divisor = 3   ->  3     (10/3 = 3.333.. -> 3)
dividend = 7,  divisor = -3  ->  -2    (7/-3 = -2.333.. -> -2)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Biểu diễn nhị phân và các phép toán bit
2. Số nguyên 32 bit, tràn số và tách chữ số

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Biểu diễn nhị phân và các phép toán bit

### Số nguyên dưới dạng bit

Máy tính lưu số nguyên ở hệ **nhị phân**: mỗi chữ số (bit) là 0 hoặc 1, bit thứ `k` (đếm từ 0, bên phải) có giá trị `2^k`.

```text
13 = 8 + 4 + 1 = 1·2³ + 1·2² + 0·2¹ + 1·2⁰ = 0b1101
bit:              3     2      1      0
```

In [1]:
x_bit = 13
print("bin(13) =", bin(x_bit), "| format 8 bit:", format(x_bit, "08b"), "| int('1101', 2) =", int("1101", 2))
print("bit thứ k của 13:", [(x_bit >> k) & 1 for k in range(4)], "(từ bit 0)")

bin(13) = 0b1101 | format 8 bit: 00001101 | int('1101', 2) = 13
bit thứ k của 13: [1, 0, 1, 1] (từ bit 0)


### Sáu phép toán bit

| Phép | Ký hiệu | Quy tắc từng bit | Ví dụ `12 = 1100`, `10 = 1010` |
|---|---|---|---|
| AND | `a & b` | 1 nếu **cả hai** là 1 | `1000` = 8 |
| OR | `a \| b` | 1 nếu **ít nhất một** là 1 | `1110` = 14 |
| XOR | `a ^ b` | 1 nếu **khác nhau** | `0110` = 6 |
| NOT | `~a` | đảo mọi bit | `~12 = -13` (Python) |
| dịch trái | `a << k` | thêm `k` số 0 bên phải = nhân `2^k` | `12 << 1 = 24` |
| dịch phải | `a >> k` | bỏ `k` bit phải = chia nguyên `2^k` | `12 >> 2 = 3` |

In [2]:
a_b, b_b = 12, 10
for ten, v in [("a & b", a_b & b_b), ("a | b", a_b | b_b), ("a ^ b", a_b ^ b_b), ("a << 1", a_b << 1), ("a >> 2", a_b >> 2), ("~a", ~a_b)]:
    print(f"{ten:<7} = {v:>4}   bin: {bin(v)}")

a & b   =    8   bin: 0b1000
a | b   =   14   bin: 0b1110
a ^ b   =    6   bin: 0b110
a << 1  =   24   bin: 0b11000
a >> 2  =    3   bin: 0b11
~a      =  -13   bin: -0b1101


### Các mẹo hay dùng

| Mục đích | Biểu thức |
|---|---|
| Kiểm tra bit `k` | `(x >> k) & 1` |
| Bật / tắt / lật bit `k` | `x \| (1 << k)`, `x & ~(1 << k)`, `x ^ (1 << k)` |
| Số chẵn? | `x & 1 == 0` |
| Bỏ bit 1 thấp nhất | `x & (x - 1)` |
| Lấy bit 1 thấp nhất | `x & -x` |
| Là lũy thừa của 2? | `x > 0 and x & (x - 1) == 0` |

**Tính chất XOR** (cực kỳ hay dùng): `x ^ x = 0`, `x ^ 0 = x`, giao hoán và kết hợp → XOR cả dãy thì các số xuất hiện **chẵn** lần triệt tiêu nhau.

In [3]:
x_m = 0b10110
print("bỏ bit 1 thấp nhất:", format(x_m, "05b"), "→", format(x_m & (x_m - 1), "05b"))
print("lấy bit 1 thấp nhất:", format(x_m & -x_m, "05b"))
print("lũy thừa của 2:", [v for v in range(1, 40) if v & (v - 1) == 0])
print("đếm bit 1 của 255:", bin(255).count("1"))
xor_all = 0
for v in [4, 1, 2, 1, 2]:
    xor_all ^= v
print("XOR [4,1,2,1,2] =", xor_all, "(số xuất hiện lẻ lần)")

bỏ bit 1 thấp nhất: 10110 → 10100
lấy bit 1 thấp nhất: 00010
lũy thừa của 2: [1, 2, 4, 8, 16, 32]
đếm bit 1 của 255: 8
XOR [4,1,2,1,2] = 4 (số xuất hiện lẻ lần)


### Dùng bit làm tập hợp (bitmask)

Với `n` phần tử, một số `mask` từ `0` tới `2^n − 1` biểu diễn một **tập con**: bit `i` bằng 1 nghĩa là lấy phần tử `i`. Duyệt mọi `mask` = duyệt mọi tập con.

In [4]:
items = ["a", "b", "c"]
for mask in range(1 << len(items)):
    tap = [items[i] for i in range(len(items)) if mask >> i & 1]
    print(format(mask, "03b"), tap)

000 []
001 ['a']
010 ['b']
011 ['a', 'b']
100 ['c']
101 ['a', 'c']
110 ['b', 'c']
111 ['a', 'b', 'c']


### Số âm và giới hạn 32 bit

Ngôn ngữ như C/Java lưu `int` bằng 32 bit, số âm dạng **bù hai** (two's complement): `-x = ~x + 1`. Python có số nguyên **không giới hạn**, nên muốn mô phỏng 32 bit phải **che** bằng `& 0xFFFFFFFF` rồi đổi lại thành số có dấu.

In [5]:
def thanh_32bit(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >> 31 else x       # bit 31 là bit dấu

print("-1 ở 32 bit:", format(-1 & 0xFFFFFFFF, "032b"))
print("2^31 tràn thành:", thanh_32bit(2 ** 31), "| (2^31 - 1) + 1 =", thanh_32bit(2 ** 31 - 1 + 1))

-1 ở 32 bit: 11111111111111111111111111111111
2^31 tràn thành: -2147483648 | (2^31 - 1) + 1 = -2147483648


Lỗi hay gặp: nhầm `^` là lũy thừa (lũy thừa là `**`); quên ngoặc vì `&`, `|`, `^` có độ ưu tiên **thấp hơn** `==` — viết `(x & 1) == 0`; `~x` trong Python là `-x - 1`, không phải "đảo 32 bit".

## 2.2 Số nguyên 32 bit, tràn số và tách chữ số

### Giới hạn 32 bit có dấu

Nhiều đề yêu cầu "môi trường chỉ lưu được số nguyên 32 bit có dấu":

```text
INT_MIN = -2^31     = -2147483648
INT_MAX =  2^31 - 1 =  2147483647
```

Vượt khỏi khoảng này gọi là **tràn số** (overflow). C/Java sẽ cho kết quả sai (quay vòng); Python thì số nguyên **không giới hạn**, nên ta phải **tự kiểm tra** để đúng yêu cầu đề.

In [6]:
INT_MIN_D, INT_MAX_D = -2 ** 31, 2 ** 31 - 1
print(INT_MIN_D, INT_MAX_D)
print("Python không tràn:", INT_MAX_D + 1, "| mô phỏng C (quay vòng):", ((INT_MAX_D + 1 + 2 ** 31) % 2 ** 32) - 2 ** 31)

def kep_32(x):
    return max(INT_MIN_D, min(INT_MAX_D, x))
print("kẹp 10^10 về:", kep_32(10 ** 10), "| kẹp -10^10:", kep_32(-10 ** 10))

-2147483648 2147483647
Python không tràn: 2147483648 | mô phỏng C (quay vòng): -2147483648
kẹp 10^10 về: 2147483647 | kẹp -10^10: -2147483648


### Tách và ghép chữ số

- Chữ số cuối: `x % 10`; bỏ chữ số cuối: `x // 10` (với `x >= 0`).
- Ghép thêm chữ số `d` vào bên phải: `x = x * 10 + d`.
- `divmod(x, 10)` trả cả hai cùng lúc.

**Cẩn thận với số âm:** trong Python `-123 // 10 = -13` và `-123 % 10 = 7` (làm tròn **xuống**). Nên xử lý **dấu riêng**, làm việc với `abs(x)`.

In [7]:
x_d = 12345
chu_so = []
while x_d > 0:
    x_d, d = divmod(x_d, 10)
    chu_so.append(d)
print("chữ số (từ phải sang):", chu_so)
y_d = 0
for d in [5, 4, 3, 2, 1]:
    y_d = y_d * 10 + d
print("ghép lại:", y_d)
print("số âm: -123 // 10 =", -123 // 10, "| -123 % 10 =", -123 % 10, "| int(-123 / 10) =", int(-123 / 10))

chữ số (từ phải sang): [5, 4, 3, 2, 1]
ghép lại: 54321
số âm: -123 // 10 = -13 | -123 % 10 = 7 | int(-123 / 10) = -12


### Kiểm tra tràn **trước khi** tràn

Khi không được dùng số lớn hơn 32 bit (như trong C), phải kiểm tra trước bước `x = x * 10 + d`:

```text
x * 10 + d > INT_MAX   ⟺   x > INT_MAX // 10   hoặc   (x == INT_MAX // 10 và d > 7)
```

(vì `INT_MAX = 214748364·10 + 7`). Trong Python có thể tính rồi so sánh, nhưng hiểu điều kiện này giúp viết đúng ở mọi ngôn ngữ.

In [8]:
def ghep_an_toan(chu_so_list):
    x = 0
    for d in chu_so_list:
        if x > INT_MAX_D // 10 or (x == INT_MAX_D // 10 and d > 7):
            return None                                   # sẽ tràn
        x = x * 10 + d
    return x

print(ghep_an_toan([2, 1, 4, 7, 4, 8, 3, 6, 4, 7]), ghep_an_toan([2, 1, 4, 7, 4, 8, 3, 6, 4, 8]))

2147483647 None


## 2.3 Áp dụng vào bài này

Không được dùng `*`, `/`, `%` → thay phép chia bằng **trừ nhiều lần**, và để nhanh thì trừ các khối **nhân đôi** `divisor << k` (dịch bit trái = nhân 2). Kết quả phải nằm trong 32 bit có dấu: trường hợp duy nhất tràn là `−2^31 / −1`.

# 3. Nền tảng

## 3.1 "Làm tròn về 0" khác `//` của Python

`//` của Python làm tròn **xuống** (về âm vô cùng). Với số âm, kết quả khác "làm tròn về 0":

| Phép | Giá trị thật | `//` (xuống) | Làm tròn về 0 |
|---|---|---|---|
| 7 / 3 | 2.33 | 2 | 2 |
| 7 / -3 | -2.33 | **-3** | **-2** |
| -7 / 3 | -2.33 | **-3** | **-2** |
| -7 / -3 | 2.33 | 2 | 2 |

Vì vậy cách an toàn: **tính trên trị tuyệt đối**, rồi gắn dấu.

In [9]:
print(7 // -3, int(7 / -3))     # -3 (xuống) và -2 (về 0)
print(-7 // 3, int(-7 / 3))

def truncate_div_reference(a: int, b: int) -> int:
    """Phép chia làm tròn về 0, dùng để KIỂM TRA (được phép dùng // ở đây)."""
    q = abs(a) // abs(b)
    return q if (a < 0) == (b < 0) else -q

print(truncate_div_reference(7, -3), truncate_div_reference(10, 3))

-3 -2
-3 -2
-2 3


## 3.2 Dịch bit: `x << k` = `x · 2^k`

`<<` dịch các bit sang trái `k` vị trí, tương đương nhân với `2^k` — nhưng **không dùng phép nhân**.

```text
3      = 0b11
3 << 1 = 0b110   = 6
3 << 2 = 0b1100  = 12
```

Tương tự `x + x` cũng là nhân đôi mà không dùng `*`.

In [10]:
for k in range(5):
    print(f"3 << {k} = {3 << k:>3}  ({bin(3 << k)})")
print(5 + 5, 5 << 1)

3 << 0 =   3  (0b11)
3 << 1 =   6  (0b110)
3 << 2 =  12  (0b1100)
3 << 3 =  24  (0b11000)
3 << 4 =  48  (0b110000)
10 10


# 4. Cách 1: trừ lặp

Chia `a` cho `b` (đều dương) = đếm xem trừ được `b` bao nhiêu lần trước khi `a < b`.

In [11]:
def divide_subtract(a: int, b: int) -> int:
    count = 0
    while a >= b:
        a -= b
        count += 1
    return count

print(divide_subtract(10, 3))
print(divide_subtract(7, 7))

3
1


**Vấn đề:** `2147483647 / 1` cần **2 tỷ** lần trừ. Thời gian `O(thương)` — quá chậm.

In [12]:
import time
t = time.time()
divide_subtract(10**6, 1)
print(f"Chia 10^6 cho 1 bằng trừ lặp mất {time.time() - t:.3f} giây — với 2^31 sẽ lâu hơn ~2000 lần")

Chia 10^6 cho 1 bằng trừ lặp mất 0.092 giây — với 2^31 sẽ lâu hơn ~2000 lần

# 5. Ý tưởng chính: trừ những "khối" nhân đôi

Thay vì trừ `b` từng lần, ta trừ **khối lớn nhất có thể** dạng `b · 2^k`.

Ví dụ `a = 100`, `b = 3`:

```text
3, 6, 12, 24, 48, 96   <- nhân đôi tới khi khối tiếp theo (192) > 100
100 - 96 = 4,  thương += 32       (96 = 3 · 32)
còn 4: khối lớn nhất ≤ 4 là 3
4 - 3 = 1,     thương += 1
còn 1 < 3: dừng.  Thương = 33
```

Đây chính là **chia dài trong hệ nhị phân**: mỗi khối `b · 2^k` ứng với bit thứ `k` của thương.

## Cách viết đơn giản

```text
while a >= b:
    block, multiple = b, 1
    while a >= block + block:      # còn nhân đôi được
        block += block
        multiple += multiple
    a -= block
    quotient += multiple
```

In [13]:
def divide_positive(a: int, b: int) -> int:
    quotient = 0
    while a >= b:
        block, multiple = b, 1
        while a >= block + block:
            block += block
            multiple += multiple
        a -= block
        quotient += multiple
    return quotient

print(divide_positive(100, 3))
print(divide_positive(10, 3))
print(divide_positive(2**31 - 1, 1))

33
3
2147483647


# 6. Mô phỏng từng bước: `100 / 3`

In [14]:
def trace_divide_positive(a: int, b: int) -> int:
    quotient = 0
    round_no = 1
    while a >= b:
        block, multiple = b, 1
        chain = [block]
        while a >= block + block:
            block += block
            multiple += multiple
            chain.append(block)
        print(f"Vòng {round_no}: a = {a:>3}, nhân đôi: {chain} -> trừ {block} (= {b} x {multiple})")
        a -= block
        quotient += multiple
        print(f"         còn a = {a}, thương = {quotient}")
        round_no += 1
    print(f"Dừng vì a = {a} < {b}. Thương = {quotient}, dư = {a}")
    return quotient

_ = trace_divide_positive(100, 3)

Vòng 1: a = 100, nhân đôi: [3, 6, 12, 24, 48, 96] -> trừ 96 (= 3 x 32)
         còn a = 4, thương = 32
Vòng 2: a =   4, nhân đôi: [3] -> trừ 3 (= 3 x 1)
         còn a = 1, thương = 33
Dừng vì a = 1 < 3. Thương = 33, dư = 1


# 7. Xử lý dấu và tràn số

1. **Tràn duy nhất:** `-2^31 / -1 = 2^31` vượt `2^31 - 1` → trả `2^31 - 1`. Mọi trường hợp khác thương nằm trong phạm vi (vì `|thương| <= |dividend|`).
2. **Dấu:** kết quả âm khi đúng **một** trong hai số âm: `negative = (dividend < 0) != (divisor < 0)`.
3. Chia trên `abs(dividend)`, `abs(divisor)`, rồi gắn dấu bằng `-quotient` (phép đổi dấu, không phải nhân).

> **Ghi chú về 32-bit:** trong Java/C++, `abs(-2^31)` **tràn** vì `2^31` không vừa `int`. Lời giải kinh điển ở các ngôn ngữ đó làm việc trên **số âm** (vì phía âm rộng hơn). Python không giới hạn số nguyên nên ta dùng `abs` cho dễ hiểu; logic thuật toán giống hệt.

In [15]:
INT_MAX, INT_MIN = 2**31 - 1, -2**31

def divide(dividend: int, divisor: int) -> int:
    if dividend == INT_MIN and divisor == -1:
        return INT_MAX

    negative = (dividend < 0) != (divisor < 0)
    a, b = abs(dividend), abs(divisor)

    quotient = 0
    while a >= b:
        block, multiple = b, 1
        while a >= block + block:
            block += block
            multiple += multiple
        a -= block
        quotient += multiple

    return -quotient if negative else quotient

for x, y in [(10, 3), (7, -3), (-7, 3), (-7, -3), (0, 5), (INT_MIN, -1), (INT_MIN, 1), (1, 2)]:
    print(f"{x:>11} / {y:>2} = {divide(x, y)}")

         10 /  3 = 3
          7 / -3 = -2
         -7 /  3 = -2
         -7 / -3 = 2
          0 /  5 = 0
-2147483648 / -1 = 2147483647
-2147483648 /  1 = -2147483648
          1 /  2 = 0


# 8. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def divide(self, dividend: int, divisor: int) -> int:
        INT_MAX, INT_MIN = 2**31 - 1, -2**31
        if dividend == INT_MIN and divisor == -1:
            return INT_MAX
        negative = (dividend < 0) != (divisor < 0)
        a, b = abs(dividend), abs(divisor)
        quotient = 0
        while a >= b:
            block, multiple = b, 1
            while a >= block + block:
                block += block
                multiple += multiple
            a -= block
            quotient += multiple
        return -quotient if negative else quotient

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS  {actual}")


sol = Solution()
check("Ví dụ 1: 10 / 3", sol.divide(10, 3), 3)
check("Ví dụ 2: 7 / -3", sol.divide(7, -3), -2)
check("Âm / dương", sol.divide(-7, 3), -2)
check("Âm / âm", sol.divide(-7, -3), 2)
check("Số bị chia = 0", sol.divide(0, 5), 0)
check("Tràn: INT_MIN / -1", sol.divide(INT_MIN, -1), INT_MAX)
check("INT_MIN / 1", sol.divide(INT_MIN, 1), INT_MIN)
check("INT_MIN / 2", sol.divide(INT_MIN, 2), -2**30)
check("INT_MAX / INT_MIN", sol.divide(INT_MAX, INT_MIN), 0)
check("INT_MIN / INT_MIN", sol.divide(INT_MIN, INT_MIN), 1)
check("Chia hết", sol.divide(81, 9), 9)

Ví dụ 1: 10 / 3              PASS  3


Ví dụ 2: 7 / -3              PASS  -2
Âm / dương                   PASS  -2
Âm / âm                      PASS  2
Số bị chia = 0               PASS  0
Tràn: INT_MIN / -1           PASS  2147483647
INT_MIN / 1                  PASS  -2147483648
INT_MIN / 2                  PASS  -1073741824
INT_MAX / INT_MIN            PASS  0
INT_MIN / INT_MIN            PASS  1
Chia hết                     PASS  9


In [18]:
import random

random.seed(29)
for _ in range(20000):
    x = random.randint(INT_MIN, INT_MAX)
    y = random.choice([random.randint(INT_MIN, INT_MAX), random.randint(-50, 50)])
    if y == 0:
        continue
    expected = truncate_div_reference(x, y)
    expected = min(expected, INT_MAX)
    assert sol.divide(x, y) == expected, (x, y)
print("20000 test ngẫu nhiên khớp phép chia làm tròn về 0")

20000 test ngẫu nhiên khớp phép chia làm tròn về 0


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** ở mọi thời điểm, `a_ban_đầu = quotient · b + a` (với `a` là phần còn lại hiện tại). Mỗi vòng trừ đi `block = b · multiple` và cộng `multiple` vào thương, nên đẳng thức được giữ. Vòng lặp dừng khi `0 <= a < b` — đúng định nghĩa thương và số dư của phép chia số nguyên dương.

**Độ phức tạp** (gọi `Q` là thương, tối đa `2^31`):

- Mỗi vòng ngoài trừ đi khối lớn nhất `b · 2^k ≤ a`, nên phần còn lại **nhỏ hơn** `b · 2^k`. Do đó vòng ngoài tiếp theo dùng `k` nhỏ hơn hẳn → vòng ngoài chạy tối đa `O(log Q)` lần.
- Mỗi vòng ngoài nhân đôi tối đa `O(log Q)` lần.

```text
Time:  O(log² Q) ≈ 31² ≈ 1000 phép tính — rất nhanh
Space: O(1)
```

Có thể đạt `O(log Q)` bằng cách nhân đôi **một lần** ra bảng `b, 2b, 4b, …` rồi duyệt từ lớn xuống nhỏ (Bài tập 5).

# 10. Những lỗi dễ mắc

## Lỗi 1: Dùng `//` trên số có dấu

`7 // -3 = -3` nhưng đáp án đúng là `-2`. Luôn chia trên trị tuyệt đối rồi gắn dấu.

## Lỗi 2: Quên trường hợp tràn `-2^31 / -1`

In [19]:
def wrong_no_overflow_check(dividend: int, divisor: int) -> int:
    negative = (dividend < 0) != (divisor < 0)
    q = divide_positive(abs(dividend), abs(divisor))
    return -q if negative else q

print("Sai :", wrong_no_overflow_check(INT_MIN, -1), "> INT_MAX =", INT_MAX)
print("Đúng:", divide(INT_MIN, -1))

Sai : 2147483648 > INT_MAX = 2147483647
Đúng: 2147483647


## Lỗi 3: Xác định dấu bằng `dividend * divisor < 0`

Vừa dùng phép nhân (bị cấm), vừa có thể tràn số trong ngôn ngữ khác. Dùng `(dividend < 0) != (divisor < 0)`.

## Lỗi 4: Điều kiện nhân đôi `block + block <= a` viết thành `block <= a` rồi nhân đôi sau

Sẽ trừ một khối **lớn hơn** `a` → `a` âm, thương sai.

In [20]:
def wrong_doubling(a: int, b: int) -> int:
    quotient = 0
    while a >= b:
        block, multiple = b, 1
        while block <= a:                   # SAI: nhân đôi quá một bước
            block += block
            multiple += multiple
        a -= block
        quotient += multiple
    return quotient

print("Sai :", wrong_doubling(10, 3), "| Đúng:", divide_positive(10, 3))

Sai : 4 | Đúng: 3


## Lỗi 5: Dùng trừ lặp

Đúng nhưng `INT_MAX / 1` sẽ chạy quá thời gian.

# 11. Mẹo nhận diện pattern

**Nhân đôi / chia đôi để giảm số bước từ `O(n)` xuống `O(log n)`:**

- Chia không dùng `/` → trừ khối `b · 2^k` (bài này).
- Lũy thừa nhanh `x^n` → bình phương liên tiếp (**#50 Pow(x, n)**).
- Nhân không dùng `*` → cộng khối `a · 2^k` (phép nhân kiểu Nga, Bài tập 4).
- Tìm kiếm khi không biết cận trên → **exponential search**.

Dấu hiệu: đề cấm phép toán nào đó, hoặc một vòng lặp "trừ / cộng từng đơn vị" có thể chạy tới `10^9` lần.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Trace `divide_positive(1000, 7)`: mỗi vòng nhân đôi tới khối nào, trừ bao nhiêu, thương tích lũy bao nhiêu?

## Bài 2 — Đoán output

```python
print(1 << 10)
print(-9 // 4, truncate_div_reference(-9, 4))
print((5 < 0) != (-2 < 0))
print(bin(33))
```

## Bài 3 — Điền chỗ trống

In [21]:
def exercise_3_divide(dividend: int, divisor: int) -> int:
    INT_MAX, INT_MIN = 2**31 - 1, -2**31
    # TODO 1: xử lý tràn INT_MIN / -1
    # TODO 2: xác định dấu, lấy a, b là trị tuyệt đối
    quotient = 0
    # TODO 3: vòng ngoài trừ khối; vòng trong nhân đôi block và multiple
    # TODO 4: gắn dấu, trả về
    pass


# assert exercise_3_divide(10, 3) == 3
# assert exercise_3_divide(7, -3) == -2
# assert exercise_3_divide(-2**31, -1) == 2**31 - 1

## Bài 4 — Biến thể: nhân không dùng `*` (phép nhân kiểu Nga)

Nhân hai số nguyên **không âm** chỉ dùng `+`, `>>`, `<<`, `&`.

Ý tưởng: `a · b = Σ (a << k)` với mọi bit `k` bằng 1 trong `b`.

```python
multiply(13, 11) -> 143
multiply(0, 5)   -> 0
```

In [22]:
def exercise_4_multiply(a: int, b: int) -> int:
    # TODO
    pass


# assert exercise_4_multiply(13, 11) == 143
# assert exercise_4_multiply(0, 5) == 0
# assert exercise_4_multiply(12345, 6789) == 12345 * 6789

## Bài 5 — Mini challenge: chia trong `O(log Q)`

Viết lại `divide_positive` sao cho chỉ nhân đôi **một lần**: lập danh sách khối `b, 2b, 4b, …` (tới khi vượt `a`), rồi duyệt danh sách **từ lớn xuống nhỏ**, trừ khối nếu được.

In [23]:
def exercise_5_divide_log(a: int, b: int) -> int:
    # TODO
    pass


# assert exercise_5_divide_log(100, 3) == 33
# assert exercise_5_divide_log(2**31 - 1, 1) == 2**31 - 1
# assert exercise_5_divide_log(5, 7) == 0

# 13. Gợi ý

**Bài 1**
- Mức 1: chuỗi nhân đôi đầu tiên: `7, 14, 28, 56, 112, 224, 448, 896`.
- Mức 2: `1000 - 896 = 104`, thương `128`.

**Bài 3**
- Mức 1: so với code mục 7.

**Bài 4**
- Mức 1: duyệt từng bit của `b` từ thấp lên: `if b & 1: result += a`.
- Mức 2: sau mỗi bước `a <<= 1`, `b >>= 1`.

**Bài 5**
- Mức 1: `blocks = [(b, 1)]`, nhân đôi tới khi khối tiếp theo `> a`.
- Mức 2: `for block, multiple in reversed(blocks): if a >= block: a -= block; q += multiple`.
- Mức 3: mỗi khối được thử **đúng một lần** vì sau khi trừ khối `b · 2^k`, phần còn lại `< b · 2^k`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
_ = trace_divide_positive(1000, 7)

Vòng 1: a = 1000, nhân đôi: [7, 14, 28, 56, 112, 224, 448, 896] -> trừ 896 (= 7 x 128)
         còn a = 104, thương = 128
Vòng 2: a = 104, nhân đôi: [7, 14, 28, 56] -> trừ 56 (= 7 x 8)
         còn a = 48, thương = 136
Vòng 3: a =  48, nhân đôi: [7, 14, 28] -> trừ 28 (= 7 x 4)
         còn a = 20, thương = 140
Vòng 4: a =  20, nhân đôi: [7, 14] -> trừ 14 (= 7 x 2)
         còn a = 6, thương = 142
Dừng vì a = 6 < 7. Thương = 142, dư = 6


## Đáp án Bài 2

In [25]:
print(1 << 10)                                   # 1024
print(-9 // 4, truncate_div_reference(-9, 4))    # -3 -2
print((5 < 0) != (-2 < 0))                       # True: khác dấu -> kết quả âm
print(bin(33))                                   # 0b100001 = 32 + 1 (giống 100/3: 32 + 1)

1024
-3 -2
True
0b100001


## Đáp án Bài 3

In [26]:
def answer_3_divide(dividend: int, divisor: int) -> int:
    INT_MAX, INT_MIN = 2**31 - 1, -2**31
    if dividend == INT_MIN and divisor == -1:
        return INT_MAX
    negative = (dividend < 0) != (divisor < 0)
    a, b = abs(dividend), abs(divisor)
    quotient = 0
    while a >= b:
        block, multiple = b, 1
        while a >= block + block:
            block += block
            multiple += multiple
        a -= block
        quotient += multiple
    return -quotient if negative else quotient

assert answer_3_divide(10, 3) == 3
assert answer_3_divide(7, -3) == -2
assert answer_3_divide(-2**31, -1) == 2**31 - 1
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_multiply(a: int, b: int) -> int:
    result = 0
    while b:
        if b & 1:
            result += a
        a <<= 1
        b >>= 1
    return result

assert answer_4_multiply(13, 11) == 143
assert answer_4_multiply(0, 5) == 0
assert answer_4_multiply(12345, 6789) == 12345 * 6789
print("Bài 4 PASS")

Bài 4 PASS


`11 = 0b1011` → `13·11 = 13·1 + 13·2 + 13·8 = 13 + 26 + 104 = 143`. Phép chia ở bài này là **phép ngược** của đúng quá trình đó.

## Đáp án Bài 5

In [28]:
def answer_5_divide_log(a: int, b: int) -> int:
    if a < b:
        return 0
    blocks = [(b, 1)]
    while blocks[-1][0] + blocks[-1][0] <= a:
        block, multiple = blocks[-1]
        blocks.append((block + block, multiple + multiple))
    quotient = 0
    for block, multiple in reversed(blocks):
        if a >= block:
            a -= block
            quotient += multiple
    return quotient

assert answer_5_divide_log(100, 3) == 33
assert answer_5_divide_log(2**31 - 1, 1) == 2**31 - 1
assert answer_5_divide_log(5, 7) == 0
for _ in range(5000):
    x, y = random.randint(0, 2**31), random.randint(1, 2**31)
    assert answer_5_divide_log(x, y) == x // y
print("Bài 5 PASS (kể cả 5000 test ngẫu nhiên)")

Bài 5 PASS (kể cả 5000 test ngẫu nhiên)


# 15. Tổng kết

> Phép chia = đếm số lần trừ. Thay vì trừ từng đơn vị `b`, trừ **khối lớn nhất** `b · 2^k` — giống chia dài trong hệ nhị phân. Tính trên trị tuyệt đối, gắn dấu sau, và nhớ ca tràn `-2^31 / -1`.

---

## Checklist trước khi nộp

- [ ] Không dùng `*`, `/`, `%` (và `//` trên số có dấu)?
- [ ] Xử lý `INT_MIN / -1 → INT_MAX`?
- [ ] Dấu: `(dividend < 0) != (divisor < 0)`?
- [ ] Nhân đôi chỉ khi `block + block <= a`?
- [ ] Đã thử `0 / x`, `x / 1`, `INT_MIN / 1`, `INT_MIN / INT_MIN`, chia hai số khác dấu?